# Regression Rating Prediction

**Objective:** Predict Rating and compare a baseline with a Random Forest model.

The code is split into visible, explainable steps for a demonstration video.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'Tourism Dataset'
PROCESSED = ROOT / 'data' / 'processed'
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
master = pd.read_csv(PROCESSED/'master_dataset.csv')
FEATURES = ['VisitYear','VisitMonth','UserContinent','UserRegion','UserCountry','UserCity','AttractionType','AttractionCity','AttractionCountry','AttractionRegion','AttractionContinent']
X, y = master[FEATURES].fillna('Unknown'), master['Rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=RANDOM_STATE)

In [ ]:
categorical = [c for c in FEATURES if c not in ['VisitYear','VisitMonth']]
preprocess = ColumnTransformer([('categorical', OneHotEncoder(handle_unknown='ignore'), categorical), ('numeric', 'passthrough', ['VisitYear','VisitMonth'])])
models = {'baseline_mean': DummyRegressor(strategy='mean'), 'random_forest': RandomForestRegressor(n_estimators=60, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)}

In [ ]:
results, trained = [], {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessing', preprocess), ('model', model)]).fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    results.append([name, mean_absolute_error(y_test,pred), mean_squared_error(y_test,pred), mean_squared_error(y_test,pred)**.5, r2_score(y_test,pred)])
    trained[name] = pipeline
comparison = pd.DataFrame(results, columns=['model','MAE','MSE','RMSE','R2']).sort_values('RMSE')
comparison

In [ ]:
best_model = trained[comparison.iloc[0]['model']]
joblib.dump(best_model, ROOT/'models'/'regression'/'rating_model.pkl')
# Streamlit clips displayed predictions to the valid 1 to 5 range.
np.clip(best_model.predict(X_test.head(5)), 1, 5)